In [1]:
# -*- coding: utf-8 -*-
"""
@author: Alain Plantec

Voici un skelette possible.
Vous devez programmez les classes contenues dans ce squelette.
Les fonctions et leurs paramètres ainsi que les variables contenues dans le classes
ont du sens par rapport à une version programmée pour la préparation du projet.
Votre version sera forcément différente.
Donc, vous pouvez ajouter/retirer des variables et/ou des fonctions et/ou des paramètres.

"""
try:  # import as appropriate for 2.x vs. 3.x
	import tkinter as tk
	import tkinter.messagebox as tkMessageBox
except:
	import Tkinter as tk
	import tkMessageBox

from sokobanXSBLevels import *
from enum import Enum

from datetime import timedelta
import json

"""
Direction :

	Utile pour gérer le calcul des positions pour les mouvements
"""
class Direction(Enum):
    Up = 1
    Down = 2
    Left = 3
    Right = 4

class Position:
    """
    Classe Position :
        - Stocke les coordonnées x et y.
        - Vérifie si x et y sont valides par rapport à une matrice.
        - Calcule une position relative à partir d'un décalage (offset) et d'une direction.
    """

    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f'Position({self.x}, {self.y})'

    # Getters
    def getX(self):
        return self.x

    def getY(self):
        return self.y

    # Setters
    def setX(self, newx):
        self.x = newx

    def setY(self, newy):
        self.y = newy

    # Retourne une nouvelle position en fonction de la direction et du décalage.
    def positionTowards(self, direction, offset):
        new_x, new_y = self.x, self.y

        if direction == Direction.Up:
            new_y -= offset
        elif direction == Direction.Down:
            new_y += offset
        elif direction == Direction.Left:
            new_x -= offset
        elif direction == Direction.Right:
            new_x += offset

        return Position(new_x, new_y)

    # Retourne True si les coordonnées sont valides dans le warehouse.
    def isValidInWharehouse(self, warehouse):
        return warehouse.isPositionValid(self)

    # Convertit la position actuelle en coordonnées pour un Canvas.
    def asCanvasPositionIn(self, elem):
        lx = self.getX() * elem.getWidth()
        ly = self.getY() * elem.getHeight()
        return Position(lx, ly)


"""
WharehousePlan : Plan de l'entrepot pour stocker les éléments.
	Les éléments sont stockés dans une matrice (#rawMatrix)
"""
class WharehousePlan:
    """
    Classe WharehousePlan :
        - Gère un plan de matrice représentant un entrepôt.
        - Permet la gestion d'éléments à des positions spécifiques.
    """
    def __init__(self):
        # La matrice de stockage des éléments (chaque élément peut être un objet Elem)
        self.rawMatrix = []

    # Ajouter une rangée à la matrice
    def appendRow(self, row):
        self.rawMatrix.append(row)

    # Récupère l'élément à une position donnée
    def at(self, position):
        x, y = position.getX(), position.getY()
        if 0 <= y < len(self.rawMatrix) and 0 <= x < len(self.rawMatrix[y]):
            return self.rawMatrix[y][x]
        return None

    # Place un élément à une position donnée
    def atPut(self, position, elem):
        x, y = position.getX(), position.getY()
        if 0 <= y < len(self.rawMatrix) and 0 <= x < len(self.rawMatrix[y]):
            self.rawMatrix[y][x] = elem

    # Vérifie si une position est valide (c'est-à-dire si elle existe dans la matrice)
    def isPositionValid(self, position):
        return self.at(position) is not None

    # Vérifie si une position contient un emplacement libre
    def hasFreePlaceAt(self, position):
        elem = self.at(position)
        return elem is not None and elem.isFreePlace()

    # Convertit le plan en une matrice XSB (représentation spécifique)
    def asXsbMatrix(self):
        # Exemple de conversion : chaque élément de rawMatrix doit avoir une méthode asXsb()
        return [[elem.asXsb() if elem else ' ' for elem in row] for row in self.rawMatrix]

"""
Floor :
	Représente une case vide de la matrice
	(pas de None dans la matrice)

"""
class Floor(object):

    def __init__(self):
        # Constructeur vide pour l'instant
        None

    # Retourne False car un sol n'est pas un élément mobile
    def isMovable(self):
        return False

    # Retourne True car un sol peut être recouvert
    def canBeCovered(self):
        return True

    # Retourne le caractère associé au sol dans un format XSB (par défaut, un espace)
    def xsbChar(self):
        return ' '

    # Retourne True car un sol est un emplacement libre
    def isFreePlace(self):
        return True
"""
Goal :
	Représente une localisation à recouvrir d'un BOX (objectif du jeu).
	Le déménageur doit parvenir à couvrir toutes ces cellules à partir des caisses.
	Un Goal est static, il est toujours déssiné en dessous :
    	Le zOrder est assuré par le tag du create_image (tag='static')
    	et self.canvas.tag_raise("movable","static") dans Level
"""
class Goal(object):
    def __init__(self, canvas, position):
        imagelink = 'goal.png'
        self.image = tk.PhotoImage(file=imagelink)
        self.width = self.image.width()
        self.height = self.image.height()
        self.position = position

        pos = self.position.asCanvasPositionIn(self)
        canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="static", anchor=tk.NW)

    def isMovable(self):
        return False

    def getHeight(self):
        return self.height

    def getWidth(self):
        return self.width

    def canBeCovered(self):
        return True

    def xsbChar(self):
        return '.'

    def isFreePlace(self):
        return False


"""
Wall : pour délimiter les murs
	Le déménageur ne peut pas traverser un mur.
	Un Wall est static, il est toujours déssiné en dessous :
    	Le zOrder est assuré par le tag du create_image (tag='static')
    	et self.canvas.tag_raise("movable","static") dans Level
"""
class Wall(object):
    def __init__(self, canvas, position):
        imagelink = 'wall.png'
        self.image = tk.PhotoImage(file=imagelink)
        self.width = self.image.width()
        self.height = self.image.height()
        self.position = position

        pos = self.position.asCanvasPositionIn(self)
        canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="static", anchor=tk.NW)

    def getHeight(self):
        return self.height

    def getWidth(self):
        return self.width

    def isMovable(self):
        return False

    def canBeCovered(self):
        return False

    def xsbChar(self):
        return '#'

    def isFreePlace(self):
        return False

"""
Box : Caisse à déplacer par le déménageur.
	Etant donné qu'une caisse doit être déplacé, le canvas et la matrice sont necessaires pour
	reconstruire l'image et mettre en oeuvre sont déplacement (dans le canvas et dans la matrice)
	Un Box est "movable", il est toujours déssiné au dessus des objets "static" :
    	Le zOrder est assuré par le tag du create_image (tag='movable')
    	et self.canvas.tag_raise("movable","static") dans Level
	Un Box est représenté differemment (image différente) suivant qu'il se situe sur un emplacement marqué par un Goal ou non.
 """
class Box(object):
    def __init__(self, canvas, wharehouse, position, onGoal):
        self.onGoal = onGoal
        if self.onGoal == True:
            imagelink = 'boxOnTarget.png'
        else:
            imagelink = 'box.png'

        self.image = tk.PhotoImage(file=imagelink)
        self.width = self.image.width()
        self.height = self.image.height()
        self.position = position
        pos = self.position.asCanvasPositionIn(self)
        self.image_id = canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)
        self.wharehouse = wharehouse
        self.canvas = canvas
        self.animation = None
        self.circleDiameter = 50
        self.on_goal=onGoal

    def getHeight(self):
        return self.height

    def getWidth(self):
        return self.width

    def isMovable(self):
        return True

    def canBeCovered(self):
        return False

    def moveTowards(self, direction):
        # Comme canMove dans le mover
        if direction == Direction.Up:
            dx, dy = 0, -1
        elif direction == Direction.Down:
            dx, dy = 0, 1
        elif direction == Direction.Left:
            dx, dy = -1, 0
        elif direction == Direction.Right:
            dx, dy = 1, 0
        else:
            dx, dy = 0, 0

        new_x = self.position.getX() + dx
        new_y = self.position.getY() + dy

        self.canvas.delete(self.image_id)
        # Créer la nouvelle image à la bonne position
        if self.onGoal == False:
            self.image = tk.PhotoImage(file='box.png')
        else:
            self.image = tk.PhotoImage(file='boxOnTarget.png')

        self.image_id = self.canvas.create_image(new_x * 64, new_y * 64, image=self.image, anchor=tk.NW, tag="movable")

        self.position.setX(new_x)
        self.position.setY(new_y)

    def xsbChar(self):
        if self.under.isFreePlace():
            return '$'
        else:
            return '*'

    def isFreePlace(self):
        return False
    
    def startGoalCoveredAnimation(self):
        self.canvas.after(100, self.goalCoveredAnimation)

    def cleanUpAnimation(self):
        self.canvas.delete(self.animation)
        self.animation = None
        self.circleDiameter = 50

    def goalCoveredAnimation(self):
        self.canvas.delete(self.animation)
        self.animation = None

        self.animation = self.canvas.create_oval(self.position.getX() * 64 , self.position.getY() * 64, self.position.getX() * 64 + self.circleDiameter, self.position.getY() * 64 + self.circleDiameter, fill="red")
        self.circleDiameter -= 0.5

        if (self.circleDiameter < 0):
            self.circleDiameter = 50

        if (self.onGoal):
            self.canvas.after(100, self.goalCoveredAnimation)



"""
Mover : C'est  le déménageur.
	La classe Mover met en oeuvre la logique du jeu dans #canMove et #moveTowards.
	Etant donné qu'un Mover se déplace, le canvas et la matrice sont necessaires pour
	reconstruire l'image et mettre en oeuvre sont déplacement (dans le canvas et dans la matrice)
	Un Mover est "movable", il est toujours déssiné au dessus des objets "static" :
    	Le zOrder est assuré par le tag du create_image (tag='movable')
    	et self.canvas.tag_raise("movable","static") dans Level
	Un Box est représenté differemment (image différente) suivant la direction de déplacement (même si le dépplacement s'avère impossible).
"""
class Mover(object):
    def __init__(self, canvas, wharehouse, position, onGoal):
        imagelink = 'playerDown.png'
        self.image = tk.PhotoImage(file=imagelink)

        self.position = position
        self.canvas = canvas
        self.height = 64
        self.width = 64
        self.wharehouse = wharehouse
        self.onGoal = onGoal
        pos = self.position.asCanvasPositionIn(self)
        self.canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)
        self.animation = None
        self.circleDiameter= 50

    def getHeight(self):
        return self.height

    def getWidth(self):
        return self.width

    def isMoveable(self):
        return True

    def moveInCanvas(self, direction):
        # Coordonnées en fonction de la direction
        if direction == Direction.Up:
            dx, dy = 0, -1
        elif direction == Direction.Down:
            dx, dy = 0, 1
        elif direction == Direction.Left:
            dx, dy = -1, 0
        elif direction == Direction.Right:
            dx, dy = 1, 0
        else:
            dx, dy = 0, 0

        # Changer la position du mover dans le canvas
        new_x = self.position.getX() * 64
        new_y = self.position.getY() * 64
        self.canvas.coords(self.image_id, new_x, new_y)

    """
    Retourne True si le Mover peut se déplacer dans la direction demandée.
    Le calcul nécessite de voir l'élément adjacent mais aussi l'élément suivant (offset de 2)
    """
    def canMove(self, direction):
        new_position = self.position.positionTowards(direction, 1)

        # Position dans les limites de la matrice
        if not new_position.isValidInWharehouse(self.wharehouse):
            return False

        # Vérifier si pas de mur
        if not self.wharehouse.hasFreePlaceAt(new_position):
            return False

        # Vérifie s'il y a une boîte dans la nouvelle position
        if self.wharehouse.at(new_position) and self.wharehouse.at(new_position).isMovable():
            # Vérifie si la boîte peut être poussée
            new_box_position = new_position.positionTowards(direction, 1)
            if not new_box_position.isValidInWharehouse(self.wharehouse) or not self.wharehouse.hasFreePlaceAt(new_box_position):
                return False

        return True

    """
    Pour le déplacement, il faut penser à déplacer éventuellement la Box et ensuite déplacer le Mover
    """
    def moveTowards(self, direction):
        # print("moveTowards")
        if self.canMove(direction):
            # print("moveTowards True")
            self.position = self.position.positionTowards(direction, 1)

            # Déplace le mover
            self.moveInCanvas(direction)

    """
    Le Mover est représenté différemment suivant la direction de déplacement
    """
    def setupImageForDirection(self, dir):
        pos = self.position.asCanvasPositionIn(self)

        if dir == Direction.Up:
            imagelink = 'playerUp.png'
            self.image = tk.PhotoImage(file=imagelink)
            self.canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)
        elif dir == Direction.Down:
            imagelink = 'playerDown.png'
            self.image = tk.PhotoImage(file=imagelink)
            self.canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)
        elif dir == Direction.Left:
            imagelink = 'playerLeft.png'
            self.image = tk.PhotoImage(file=imagelink)
            self.canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)
        elif dir == Direction.Right:
            imagelink = 'playerRight.png'
            self.image = tk.PhotoImage(file=imagelink)
            self.canvas.create_image(pos.getX(), pos.getY(), image=self.image, tags="movable", anchor=tk.NW)

    """
    Pour le déplacement :
    - Image changée en fonction de la direction
    - Si on ne peut pas se déplacer dans cette direction -> abandon
    - Sinon, le Mover est déplacé
    """
    def push(self, direction):
        self.setupImageForDirection(direction)
        if not self.canMove(direction):
            self.startImpossiblePushAnimation()
            return
        self.moveTowards(direction)

    def xsbChar(self):
        if self.under.isFreePlace():
            return '@'
        else:
            return '+'

    def isFreePlace(self):
        return False
    
    def startImpossiblePushAnimation(self):
        self.canvas.after(1, self.impossiblePushAnimation)

    def cleanUpAnimation(self):
        self.canvas.delete(self.animation)
        self.animation = None
        self.circleDiameter = 50

    def impossiblePushAnimation(self):
        if (self.circleDiameter < 0):
            self.cleanUpAnimation()
            return   

        self.canvas.delete(self.animation)
        self.animation = self.canvas.create_oval(self.position.getX() * 64 , self.position.getY() * 64, self.position.getX() * 64 + self.circleDiameter, self.position.getY() * 64 + self.circleDiameter, outline="red")
        self.circleDiameter -= 0.5
        
        self.canvas.after(1, self.startImpossiblePushAnimation)

"""
	Le jeux avec tout ce qu'il faut pour dessiner et stocker/gérer la matrice d'éléments

"""
class Level(object):
    def __init__(self, root, xsbMatrix,socobaninstance,menuinstance):
        self.root = root
        self.wharehouse = WharehousePlan()
        self.won=False
        self.socobaninstance=socobaninstance
        self.menuinstance=menuinstance

        # Calcul des dimensions de la matrice
        self.nbrows = len(xsbMatrix)
        self.nbcolumns = 0

        for line in xsbMatrix:
            nbc = len(line)
            if nbc > self.nbcolumns:
                self.nbcolumns = nbc

        self.height = self.nbrows * 64
        self.width = self.nbcolumns * 64

        self.canvas = tk.Canvas(self.root, width=self.width, height=self.height, bg="gray")
        self.canvas.pack()

        self.xsbMatrix = xsbMatrix
        self.initWharehouseFromXsb(xsbMatrix, self.nbrows, self.nbcolumns, self.wharehouse)
        self.keypressed_counter = 0
        
        # Create a Label for displaying the key press counter
        self.key_counter_label = tk.Label(self.root, text="Key Presses: 0", font=("Helvetica", 16))
        self.key_counter_label.pack(pady=10)  # Adjust position as needed
        self.root.bind("<Key>", self.keypressed)

    def initWharehouseFromXsb(self, xsbMatrix, nbrows, nbcolumns, wharehouse):
        # Légende : 
        #   '#' = wall, '$' = box, '.' = goal, '*' = box on goal, '@' = mover, '+' = mover on goal, 
        #   '-' = floor, ' ' = floor
        self.staticMatrix = []
        self.movableMatrix = []
        #rempli la marice par none
        for lineIdx in range(nbrows):
            self.staticMatrix.append([])
            self.movableMatrix.append([])
            for elemIdx in range(nbcolumns):
                self.staticMatrix[lineIdx].append(None)
                self.movableMatrix[lineIdx].append(None)

        y = 0
        for lineIdx in range(len(xsbMatrix)):
            x = 0
            for elemIdx in range(len(xsbMatrix[lineIdx])):
                e = xsbMatrix[lineIdx][elemIdx]
                if e == '#':
                    wall = Wall(self.canvas, Position(elemIdx, lineIdx))
                    self.staticMatrix[y][x] = wall
                    self.wharehouse.atPut(Position(elemIdx, lineIdx), wall)
                elif e == '@':
                    mover = Mover(self.canvas, self.wharehouse, Position(elemIdx, lineIdx), onGoal=False)
                    self.mover = mover
                    # to test 
                    self.movableMatrix[y][x] = mover
                    self.wharehouse.atPut(Position(elemIdx, lineIdx), mover)
                    self.moverX = x
                    self.moverY = y
                elif e == '$':
                    box = Box(self.canvas, self.wharehouse, Position(elemIdx, lineIdx), onGoal=False)
                    self.movableMatrix[y][x] = box
                    self.wharehouse.atPut(Position(elemIdx, lineIdx), box)
                elif e == '.':
                    goal = Goal(self.canvas, Position(elemIdx, lineIdx))
                    self.staticMatrix[y][x] = goal
                    self.wharehouse.atPut(Position(elemIdx, lineIdx), goal)
                elif e == '*':
                    box_on_goal = Box(self.canvas, self.wharehouse, Position(elemIdx, lineIdx), onGoal=True)
                    goal = Goal(self.canvas, Position(elemIdx, lineIdx))
                    self.movableMatrix[y][x] = box_on_goal
                    self.staticMatrix[y][x] = goal
                    self.wharehouse.atPut(Position(elemIdx, lineIdx), box_on_goal)
                else:
                    self.staticMatrix[y][x] = None
                    self.movableMatrix[y][x] = None
                x += 1
            y += 1
            x = 0

        self.canvas.tag_raise("movable", "static")

  

    def keypressed(self, event):
        self.keypressed_counter += 1
        self.key_counter_label.config(text=f"Key Presses: {self.keypressed_counter}")
       

        # Adapter les coordonnées du player en fonction du déplacement vouluf
        x, y = self.moverX, self.moverY
        # print("AVANT mouvement")
        # print(self.mover.position)
        direction = None
        if event.keysym == 'Up':
            direction = Direction.Up
        elif event.keysym == 'Down':
            direction = Direction.Down
        elif event.keysym == 'Left':
            direction = Direction.Left
        elif event.keysym == 'Right':
            direction = Direction.Right

        newpos = self.mover.position.positionTowards(direction, 1)
        posY, posX = newpos.getY(), newpos.getX()

        if isinstance(self.staticMatrix[posY][posX], Wall) == False:
            self.moveMover(x, y, newpos, direction)
        else:
            self.moverX = x
            self.moverY = y
            self.canvas.delete(self.mover)
            self.mover.setupImageForDirection(direction)
            self.mover.startImpossiblePushAnimation()
            
        self.checkWinCondition()

       # print("APRES mouvement")
       # print(self.mover.position)
        

    def moveMover(self, x, y, new_pos, direction):
        posY, posX = new_pos.getY(), new_pos.getX()
        # self.movableMatrix[self.moverY][self.moverX] = self.movableMatrix[y][x]
        self.movableMatrix[y][x] = None

        # Si la nouvelle position est remplie par une boite, on essaye de bouger la boite
        if (isinstance(self.movableMatrix[posY][posX], Box)):

            # La nouvelle position potentielle de la boite
            new_box_position = new_pos.positionTowards(direction,1)
            newBoxX, newBoxY = new_box_position.getX(), new_box_position.getY()

            # Si la boite est bloquee par un mur ou une autre boite, le joueur ne se deplace pas et on retourne de la fonction
            if (isinstance(self.staticMatrix[newBoxY][newBoxX],Wall) or isinstance(self.movableMatrix[newBoxY][newBoxX], Box)):
                self.mover.startImpossiblePushAnimation()
                return

            # Si la position suivante de la boite est un goal, le booleen onGoal devient True
            if (isinstance(self.staticMatrix[newBoxY][newBoxX],Goal)):
                box = Box(self.canvas, self.wharehouse, new_box_position, onGoal=True)
                # box.startGoalCoveredAnimation()
            else:
                box = Box(self.canvas, self.wharehouse, new_box_position, onGoal=False)
                # box.cleanUpAnimation()
            
            # On deplace la boite
            self.movableMatrix[y][x] = None
            self.movableMatrix[newBoxY][newBoxX] = box
            self.movableMatrix[posY][posX].moveTowards(direction)


        self.mover.position = new_pos
        self.moverX, self.moverY = self.mover.position.getX(), self.mover.position.getY()
        self.mover.moveTowards(direction)
        self.canvas.delete(self.mover)
        self.mover.setupImageForDirection(direction)
        self.canvas.tag_raise("movable", "static")

    def checkWinCondition(self):
        # Vérifie si toutes les boîtes sont sur les objectifs
        all_boxes_on_goals = all(isinstance(elem, Box) and elem.on_goal for row in self.movableMatrix for elem in row if isinstance(elem, Box))
        if all_boxes_on_goals:
            self.won=True
            tkMessageBox.showinfo("Victoire!", "Félicitations, vous avez terminé le niveau!")
            self.socobaninstance.showScor()
            
        if all_boxes_on_goals:
            response = tkMessageBox.askyesno("Victoire!", " Voulez-vous passer au niveau suivant?")
            if response:
                self.socobaninstance.next_level()
                
        

    

class Menu(object):
    def __init__(self):
        self.root=tk.Tk()
        self.root.title("Menu")
        text=tk.Label(self.root,text='Helooooo!',bd=100, font=('Comic Sans MS',48),bg="#ADD8E6")
        text.pack()
        tk.Label(self.root,text="Entrez le numéro du niveau de  :",font=("Comic Sans MS", 16),bg="#ADD8E6").pack(pady=10)
        self.level_entry = tk.Entry(self.root,font=("Comic Sans MS", 14),justify="center")
        self.level_entry.pack(pady=10)
        tk.Label(self.root,text="Entrez le nom du joueur  :",font=("Comic Sans MS", 16),bg="#ADD8E6").pack(pady=10)
        self.player_name = tk.StringVar()
        self.level_player = tk.Entry(self.root,font=("Comic Sans MS", 14),justify="center", textvariable = self.player_name)
        self.level_player.pack(pady=10)
        start_button = tk.Button(self.root, text="SOKOBANER",font=('Georgia',24), command=self.choseLevl,bg="#FFA07A")
        start_button.pack(pady=10)
    def choseLevl(self):
        selected_level_str = self.level_entry.get()
        self.selected_level=int(selected_level_str)
        if 0 <= self.selected_level <= len(SokobanXSBLevels)-1:
            ## Ferme la fenêtre principale pour lancer le niveau sélectionné
            self.root.destroy()
            self.new_game=Sokoban(self.selected_level,self)
            self.new_game.play()
        else:
            tkMessageBox.showinfo("STOOOOOP!",str(selected_level)+"is NOT VALIDE :/")
    def get_name_value(self):
        return self.player_name.get()
        

class Score(object):
    def __init__(self, player,scor):
        self.player=player
        self.score=scor
    def get_player(self):
        return self.player
    def get_score(self):
        return self.score
    def __str__(self):
         return str(self.player) + "["+ str(self.score)+"]"
class Librairie(object):
    def __init__(self):
        self.lesSCORE=[Score('boot',100000)]
    def ajout(self, score):
        self.lesSCORE.append(score)
    def __str__(self):
        chaine=str(self.lesSCORE[0])
        for e in self.lesSCORE[1:]:
            chaine=chaine+ "," + str(e)
        return chaine
    
    @classmethod
    def fromFile(cls,fich):
        f = open(fich,"r")
        #chargement
        tmp = json.load(f)
    
        
        liste = []
        for d in tmp:
            #créer un livre
            l=Score(d["joueur"],d["Score"])
            #l'ajouter dans la liste
            liste.append(l)
        lib=Librairie()
        lib.lesSCORE=liste
        f.close();
        print(lib.lesSCORE[1].player)
        return lib

    def toFile(self,fich):
        f = open(fich,"w")
        tmp = []
        for l in self.lesSCORE:
        #créer un dictionnaire
            d = {}
            d["joueur"] = l.player
            d["Score"] = l.score
            tmp.append(d)
        json.dump(tmp,f)
        f.close();



class Sokoban(object):
    '''
    Main Level class
    '''

    def __init__(self,level,menu):
        self.root = tk.Tk()
        self.root.resizable(True, True)
        self.root.title("Sokoban")
        self.chosen_level=level
        self.menu=menu
        self.level = Level(self.root, SokobanXSBLevels[self.chosen_level],self,self.menu)
        # print('Sokoban: ' + str(len(SokobanXSBLevels)) + ' levels')

         # Chronometer variables
        self.start_time = 0
        self.running = False

        # Create a Label for the chronometer
        self.chronometer_label = tk.Label(self.root, text="Time: 00:00", font=("Helvetica", 16))
        self.chronometer_label.pack(pady=10)  # Adjust position

        # Start the chronometer
        self.running = True
        self.update_chronometer()


        # Initialisation du niveau avec le 100e niveau (index 99)
       
      

    def loadLevel(self,levelIndex):
        self.root.after_cancel(self.timer_id)
        self.root.destroy()
        self.root = tk.Tk()
        self.root.resizable(True, True)
        self.root.title("Sokoban")
        self.level = Level(self.root, SokobanXSBLevels[levelIndex],self,self.menu)
        self.start_time = 0
        self.running = False
        self.chronometer_label = tk.Label(self.root, text="Time: 00:00", font=("Helvetica", 16))
        self.chronometer_label.pack(pady=10)
        self.running = True
        self.update_chronometer()
        self.play()
    def update_chronometer(self):
        """Update the chronometer display."""
        if self.running:
            self.start_time += 1
            self.elapsed_time = timedelta(seconds=self.start_time)
            self.chronometer_label.config(text=f"Time: {self.elapsed_time}")
            # Schedule the update_chronometer method to run after 1 second
            self.timer_id = self.root.after(1000, self.update_chronometer)

        if self.level.won:
            self.running = False
            self.root.after_cancel(self.timer_id)
            
            
    def  showScor(self):
        self.vitesse=self.level.keypressed_counter*100//self.elapsed_time.total_seconds()
        tkMessageBox.showinfo("Victoire!"+str(self.menu.get_name_value()),str(self.vitesse)+"  (pas/s)*100")
        self.scor=Score(str(self.menu.get_name_value()),str(self.vitesse))
        self.AllScores=Librairie()
        self.AllScores=self.AllScores.fromFile("Les_SCORE.json")
        self.AllScores.ajout(self.scor)
        self.AllScores.toFile("Les_SCORE.json")
        
    def next_level(self):
        if self.chosen_level + 1 < len(SokobanXSBLevels):
            self.chosen_level += 1
            self.loadLevel(self.chosen_level)
            
    def play(self):
        self.root.mainloop()

    


# Lancer le jeu
menu=Menu()
menu.root.mainloop()






Karim
